# Canonical FI names — `pkg_fi_canonical.ipynb`

**Why.** The FI name in the staging data (`cpty_fin_entity_name`, carried into `pkg_cpty_month` / `pkg_cpty_dim`) is not
one name per bank. 232 routing numbers carry 2–3 names, mostly acquisitions (COMERICA → FIFTH THIRD) and spelling
variants (GECU / GECU FEDERAL CREDIT UNION), plus typos (ALLLY BANK). Our bank metrics count distinct FI *names*, so:
- a customer paying two accounts under one routing number can count two banks;
- a routing number whose name changes shows up as "lost a bank, gained a bank" — false bank churn, in the
  `fin_out_n` / bank-churn family that drives the attrition model.

**What this builds** (small reference tables in cdi_tm; set `WRITE_TABLES = False` to only look):
1. `pkg_fi_routing` — one row per 9-digit routing number: canonical name = the name in the **latest** month the routing
   appears (acquired banks map to their current owner, consistently across history), spelling-normalised; plus every
   name seen, change type and switch month.
2. `pkg_fi_name_map` — raw FI name → canonical name, for ids without a routing prefix (name-hash ids on 3B/3C/7B/9B).

**How to use downstream:** canonical FI of a counterparty = `pkg_fi_routing.fi_canonical` via its routing prefix, else
`pkg_fi_name_map.fi_canonical` via its raw FI name, else the raw name. §5 has the join.

**Impact (§4):** for one month, the number of distinct banks per customer and direction (all rails, and ACH-only out =
the attrition `fin_out_n` definition) and month-over-month lost banks, with raw vs canonical names.

In [ ]:
# §0  Setup
import re, time, warnings, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("pkg_fi_canonical")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", "2000")

DB           = "bdahd01p_dlcdi1_cdi_tm"
T_CPTY_MONTH = f"{DB}.pkg_cpty_month"
T_CPTY_DIM   = f"{DB}.pkg_cpty_dim"
T_EDGE_M     = f"{DB}.pkg_edge_monthly"
T_CUST_DIM   = f"{DB}.pkg_cust_dim"
T_FI_ROUTING = f"{DB}.pkg_fi_routing"
T_FI_NAMEMAP = f"{DB}.pkg_fi_name_map"
WRITE_TABLES = True

IMPACT_MONTH = "2026-08"          # §4 compares this month with the previous one
FI_FIXES = {"ALLLY BANK": "ALLY BANK"}     # known typos in the FI reference names (raw → corrected)
# Words dropped to group spellings of one bank (GECU FEDERAL CREDIT UNION = GECU; WEBSTER BANK N A = WEBSTER BANK)
FI_STRIP = ["THE", "NA", "NATIONAL ASSOCIATION", "INC", "CORP", "CORPORATION", "CO", "COMPANY", "FSB", "SSB",
            "FEDERAL CREDIT UNION", "CREDIT UNION", "FCU", "LLC"]

def fi_key(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return None
    k = re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).strip()
    k = re.sub(r"\b([A-Z]) (?=[A-Z]\b)", r"\1", k)                    # N A → NA
    for w in sorted(FI_STRIP, key=len, reverse=True):
        k = re.sub(rf"\b{w}\b", " ", k)
    k = re.sub(r"\s+", " ", k).strip()
    return k or None

def prev_month(m):
    y, mm = map(int, m.split("-"))
    return f"{y - (mm == 1):04d}-{(mm - 2) % 12 + 1:02d}"

# ---- display ----
def _fmt(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if kind == "pct":
        return "0" if abs(v) < 5e-7 else (f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}")
    return f"{v:,.0f}" if kind == "int" else f"{v:,.1f}" if kind == "f1" else f"{v:,.2f}"

def fmt_df(df):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s):
            continue
        if not pd.api.types.is_numeric_dtype(s):
            out[c] = s.where(s.notna(), "")
            continue
        kind = ("pct" if "%" in str(c) else "f1" if "$M" in str(c) or "$B" in str(c)
                else "int" if (s.dropna() % 1 == 0).all() else "f2")
        out[c] = [_fmt(v, kind) for v in s]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#8a8984;margin:2px 0}</style>")

def show(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        else:
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1]).to_html(index=False, escape=True, border=0, classes="pkgt"))
    display(HTML(_CSS + "<div style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>" + "".join(parts) + "</div>"))

def note(t):
    return ("__note__", t)

ddt = dict(spark.table(T_CPTY_DIM).dtypes)
DIM_FI = next(c for c in ["fi", "cpty_fin_entity_name", "dominant_fi"] if c in ddt)
print(f"Spark {spark.version} | dim FI column: {DIM_FI} | impact month {IMPACT_MONTH} vs {prev_month(IMPACT_MONTH)}")

## §1 Routing × month × FI name (small: collected to pandas)

In [ ]:
# §1  Every FI name seen under each routing number, by month
cm = spark.table(T_CPTY_MONTH)
rt_raw = (cm.where(F.col("fi").isNotNull())
            .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
            .where(F.col("routing") != "")
            .groupBy("routing", "month", "fi")
            .agg(F.sum(F.col("amount").cast("double")).alias("amt"), F.countDistinct("cpty_id").alias("ids"))
            .toPandas())
rt_raw["fi_fixed"] = rt_raw.fi.map(lambda s: FI_FIXES.get(s, s))
rt_raw["key"] = rt_raw.fi_fixed.map(fi_key)

# all raw FI names in the dim (includes names only seen on ids without a routing prefix)
dim_names = (spark.table(T_CPTY_DIM).where(F.col(DIM_FI).isNotNull())
                  .groupBy(F.col(DIM_FI).alias("fi")).agg(F.count("*").alias("ids")).toPandas())
dim_names["fi_fixed"] = dim_names.fi.map(lambda s: FI_FIXES.get(s, s))
dim_names["key"] = dim_names.fi_fixed.map(fi_key)

show("§1 Inputs",
     ("Size", pd.DataFrame([
         {"item": "routing × month × FI rows", "value": f"{len(rt_raw):,}"},
         {"item": "routing numbers", "value": f"{rt_raw.routing.nunique():,}"},
         {"item": "months", "value": f"{rt_raw.month.min()} → {rt_raw.month.max()} ({rt_raw.month.nunique()})"},
         {"item": "distinct raw FI names under a routing", "value": f"{rt_raw.fi.nunique():,}"},
         {"item": f"distinct raw FI names in {T_CPTY_DIM}", "value": f"{len(dim_names):,}"},
         {"item": "spelling groups (normalised key)", "value": f"{dim_names.key.nunique():,}"}])))

## §2 Canonical name per routing number

- Spelling groups: names with the same normalised key share one display spelling (the one with the most dollars).
- Canonical name of a routing number = the spelling group of its dominant name (by dollars) in the **latest** month it
  appears.
- Change type: `single name`; `spelling variant` (several raw names, one key); `renamed / acquired` (keys follow each
  other in time — the old name's last month is before the new name's first month); `concurrent names` (overlapping).

In [ ]:
# §2  Build the routing table
key_amt = (pd.concat([rt_raw.groupby(["key", "fi_fixed"]).amt.sum().reset_index(),
                      dim_names.assign(amt=0.0)[["key", "fi_fixed", "amt"]]])
             .groupby(["key", "fi_fixed"]).amt.sum().reset_index()
             .sort_values(["key", "amt"], ascending=[True, False]))
KEY_SPELL = key_amt.drop_duplicates("key").set_index("key").fi_fixed.to_dict()

last_m = rt_raw.groupby("routing").month.max().rename("last_month")
first_m = rt_raw.groupby("routing").month.min().rename("first_month")
latest = (rt_raw.merge(last_m, left_on=["routing", "month"], right_on=["routing", "last_month"])
                .sort_values(["routing", "amt"], ascending=[True, False]).drop_duplicates("routing"))
latest = latest.set_index("routing")

kspan = rt_raw.groupby(["routing", "key"]).agg(k_first=("month", "min"), k_last=("month", "max"), k_amt=("amt", "sum"))
rows = []
for rt, g in rt_raw.groupby("routing"):
    keys = g.key.dropna().unique()
    raw = sorted(g.fi.unique())
    lk = latest.loc[rt, "key"]
    canon = KEY_SPELL.get(lk, latest.loc[rt, "fi_fixed"])
    if len(raw) == 1:
        ctype, switch = "single name", None
    elif len(keys) == 1:
        ctype, switch = "spelling variant", None
    elif lk is None or lk not in kspan.loc[rt].index:
        ctype, switch = "concurrent names", None
    else:
        ks = kspan.loc[rt]
        new_first = ks.loc[lk, "k_first"]
        others_last = ks.drop(index=lk).k_last.max()
        ctype = "renamed / acquired" if others_last < new_first else "concurrent names"
        switch = new_first if ctype == "renamed / acquired" else None
    rows.append({"routing": rt, "fi_canonical": canon, "n_names": len(raw), "n_keys": len(keys),
                 "change_type": ctype, "switch_month": switch, "names_seen": " | ".join(raw[:6]),
                 "first_month": first_m[rt], "last_month": last_m[rt],
                 "ids": int(g.groupby("month").ids.max().max()), "amt": float(g.amt.sum())})
ROUT = pd.DataFrame(rows)

ct = ROUT.groupby("change_type").agg(routing_numbers=("routing", "count"), amt=("amt", "sum")).reset_index()
ct["% routing numbers"] = 100 * ct.routing_numbers / ct.routing_numbers.sum()
ct["% $"] = 100 * ct.amt / ct.amt.sum()
top_change = (ROUT[ROUT.change_type != "single name"].sort_values("amt", ascending=False).head(25)
                  .assign(**{"$B": lambda d: d.amt / 1e9}))
show(f"§2 Routing numbers: {len(ROUT):,}",
     ("By change type", ct[["change_type", "routing_numbers", "% routing numbers", "% $"]]),
     ("Top 25 routing numbers with more than one name (by $)",
      top_change[["routing", "change_type", "switch_month", "fi_canonical", "names_seen", "$B"]]),
     note(f"Typo fixes applied: {FI_FIXES}. Add more to FI_FIXES and rerun from §1."))

## §3 Raw name → canonical name

In [ ]:
# §3  Name map: each raw FI name → the canonical name of the routing numbers it appears under (by $),
#     else its spelling group
rt_c = rt_raw.merge(ROUT[["routing", "fi_canonical"]], on="routing")
by_name = (rt_c.groupby(["fi", "fi_canonical"]).amt.sum().reset_index()
               .sort_values(["fi", "amt"], ascending=[True, False]).drop_duplicates("fi"))
NAMEMAP = dim_names[["fi", "fi_fixed", "key", "ids"]].merge(by_name[["fi", "fi_canonical"]], on="fi", how="left")
NAMEMAP["source"] = np.where(NAMEMAP.fi_canonical.notna(), "routing", "spelling group")
NAMEMAP["fi_canonical"] = NAMEMAP.fi_canonical.fillna(NAMEMAP.key.map(KEY_SPELL)).fillna(NAMEMAP.fi_fixed)
NAMEMAP["changed"] = NAMEMAP.fi != NAMEMAP.fi_canonical

summ = pd.DataFrame([
    {"item": "raw FI names", "value": f"{len(NAMEMAP):,}"},
    {"item": "canonical names", "value": f"{NAMEMAP.fi_canonical.nunique():,}"},
    {"item": "raw names mapped to a different name", "value": f"{NAMEMAP.changed.sum():,}"},
    {"item": "ids (dim) whose FI name changes", "value": f"{NAMEMAP.loc[NAMEMAP.changed, 'ids'].sum():,} of {NAMEMAP.ids.sum():,}"}])
groups = (NAMEMAP[NAMEMAP.changed].groupby("fi_canonical")
                 .agg(raw_names=("fi", lambda s: " | ".join(sorted(s)[:5])), n=("fi", "count"), ids=("ids", "sum"))
                 .reset_index().sort_values("ids", ascending=False).head(25))
show("§3 Name map", ("Summary", summ),
     ("Largest renames (canonical ← raw names)", groups.rename(columns={"n": "raw names folded", "ids": "ids affected"})))

## §4 Impact on bank metrics for one month

Per customer × direction in `IMPACT_MONTH`: distinct banks with raw names vs canonical names, all rails and ACH-only
outgoing (`fin_out_n`). Then month-over-month **lost banks** (bank in the previous month, not this month): with raw names
an acquisition or spelling flip shows as lost + new; with canonical names it does not. FI per counterparty is taken from
`pkg_cpty_dim`, as in the customer metrics build.

In [ ]:
# §4  Raw vs canonical bank counts and lost banks
M0, M1 = prev_month(IMPACT_MONTH), IMPACT_MONTH
rt_df = F.broadcast(spark.createDataFrame(ROUT[["routing", "fi_canonical"]]).withColumnRenamed("fi_canonical", "fi_c_rt"))
nm_df = F.broadcast(spark.createDataFrame(NAMEMAP[["fi", "fi_canonical"]]).withColumnRenamed("fi_canonical", "fi_c_nm"))

e = (spark.table(T_EDGE_M).where(F.col("month").isin(M0, M1))
          .where((F.col("src_kind") == "CPTY") | (F.col("dst_kind") == "CPTY"))
          .select("month",
                  F.when(F.col("dst_kind") == "CPTY", F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
                  F.when(F.col("dst_kind") == "CPTY", F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
                  F.when(F.col("dst_kind") == "CPTY", F.lit("out")).otherwise(F.lit("in")).alias("dir"),
                  F.substring("category", 1, 1).isin("1", "2", "3").alias("ach"))
          .distinct())
dim = spark.table(T_CPTY_DIM).select("cpty_id", F.col(DIM_FI).alias("fi")).where(F.col("fi").isNotNull())
cb = (e.join(dim, "cpty_id", "inner")
       .withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
       .join(rt_df, "routing", "left").join(nm_df, "fi", "left")
       .withColumn("fi_c", F.coalesce("fi_c_rt", "fi_c_nm", "fi"))
       .select("month", "cust", "dir", "ach", "fi", "fi_c")
       .cache())

def counts(df, ach_only):
    d = df.where(F.col("ach")) if ach_only else df
    return d.groupBy("cust", "dir").agg(F.countDistinct("fi").alias("raw"), F.countDistinct("fi_c").alias("canon"))

cd = spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("cust"), F.col("is_tm").cast("boolean").alias("is_tm"))
rows = []
for label, ach_only, dirs in [("all rails, out", False, ["out"]), ("all rails, in", False, ["in"]),
                              ("ACH out (fin_out_n)", True, ["out"])]:
    c = counts(cb.where((F.col("month") == M1) & F.col("dir").isin(dirs)), ach_only).join(cd, "cust", "left")
    r = (c.groupBy(F.coalesce("is_tm", F.lit(False)).alias("tm"))
          .agg(F.count("*").alias("customers"),
               F.avg((F.col("raw") != F.col("canon")).cast("double")).alias("chg"),
               F.avg(F.col("raw") - F.col("canon")).alias("mean_diff"),
               F.max(F.col("raw") - F.col("canon")).alias("max_diff"),
               F.avg("raw").alias("mean_raw")).toPandas())
    for _, x in r.iterrows():
        rows.append({"metric": label, "customers": "TM" if x.tm else "non-TM", "n": x.customers,
                     "% customers whose count changes": 100 * x.chg, "mean banks (raw)": x.mean_raw,
                     "mean overcount": x.mean_diff, "max overcount": x.max_diff})
cnt_tab = pd.DataFrame(rows)

# lost banks M0 → M1, per customer × direction
def lost(col):
    p0 = cb.where(F.col("month") == M0).select("cust", "dir", F.col(col).alias("b")).distinct()
    p1 = cb.where(F.col("month") == M1).select("cust", "dir", F.col(col).alias("b")).distinct()
    active = p1.select("cust", "dir").distinct()
    return (p0.join(p1, ["cust", "dir", "b"], "left_anti").join(active, ["cust", "dir"], "inner")
              .groupBy("cust", "dir").agg(F.count("*").alias(f"lost_{col}")))
L = (lost("fi").join(lost("fi_c"), ["cust", "dir"], "full").fillna(0, ["lost_fi", "lost_fi_c"]))
lt = (L.groupBy("dir").agg(F.sum((F.col("lost_fi") > 0).cast("int")).alias("with lost bank (raw)"),
                           F.sum((F.col("lost_fi_c") > 0).cast("int")).alias("with lost bank (canonical)"),
                           F.sum("lost_fi").alias("lost banks (raw)"), F.sum("lost_fi_c").alias("lost banks (canonical)"),
                           F.sum(((F.col("lost_fi") > 0) & (F.col("lost_fi_c") == 0)).cast("int")).alias("false lost only"))
        .toPandas())
lt["% lost events that are false"] = 100 * (1 - lt["lost banks (canonical)"] / lt["lost banks (raw)"].clip(lower=1))

# which raw names produce the false lost events
p0 = cb.where(F.col("month") == M0).select("cust", "dir", "fi", "fi_c").distinct()
p1 = cb.where(F.col("month") == M1).select("cust", "dir", "fi", "fi_c").distinct()
false_lost = (p0.join(p1.select("cust", "dir", "fi"), ["cust", "dir", "fi"], "left_anti")
                .join(p1.select("cust", "dir", "fi_c").distinct(), ["cust", "dir", "fi_c"], "inner")
                .groupBy("fi", "fi_c").agg(F.count("*").alias("false lost events"))
                .orderBy(F.desc("false lost events")).limit(20).toPandas())
show(f"§4 Impact, {M1} (lost banks vs {M0})",
     ("Distinct banks per customer: raw vs canonical", cnt_tab),
     ("Lost banks month over month (customers active in both months)", lt),
     ("Raw names behind the false lost events (raw name lost, canonical bank still there)",
      false_lost.rename(columns={"fi": "raw name lost", "fi_c": "canonical bank (still present)"})))

## §5 Write the reference tables

In [ ]:
# §5  Write pkg_fi_routing and pkg_fi_name_map (small, non-partitioned) and print the downstream join
if WRITE_TABLES:
    rt_out = ROUT.drop(columns=["amt"]).assign(amount=ROUT.amt)
    nm_out = NAMEMAP[["fi", "fi_canonical", "source", "ids"]].rename(columns={"fi": "fi_raw"})
    for t, pdf in [(T_FI_ROUTING, rt_out), (T_FI_NAMEMAP, nm_out)]:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
        spark.createDataFrame(pdf.astype({c: "object" for c in pdf.columns if pdf[c].dtype == object})) \
             .coalesce(1).write.mode("overwrite").format("parquet").saveAsTable(t)
    written = pd.DataFrame([{"table": T_FI_ROUTING, "rows": len(rt_out)}, {"table": T_FI_NAMEMAP, "rows": len(nm_out)}])
else:
    written = pd.DataFrame([{"table": "(not written: WRITE_TABLES = False)", "rows": 0}])
show("§5 Reference tables", ("Written", written),
     note("Canonical FI per counterparty: "
          "coalesce(pkg_fi_routing.fi_canonical via regexp_extract(cpty_id, '^(\\\\d{9})-', 1), "
          "pkg_fi_name_map.fi_canonical via the raw FI name, raw FI name)."),
     note(f"Impala: INVALIDATE METADATA {T_FI_ROUTING}; INVALIDATE METADATA {T_FI_NAMEMAP};"))